# Carga de transacciones

Ingesta horaria de transacciones de canales digitales.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso: objetivo, version, responsable y tablas involucradas.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Canales - Transacciones
# PROCESO        : ETL_TRANSACCIONES
# OBJETIVO       : Consolidar las transacciones de canales digitales
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 28/08/2026
# TABLA FUENTE   : mb_silver_prod.can.h_transaccion
# TABLA DESTINO  : mb_gold_prod.canales.fct_transaccion
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Librerias estandar, de terceros y locales, en ese orden.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parametros
> **Descripcion:** Parametros de ejecucion recibidos por widgets, para que el mismo codigo corra en cualquier ambiente.

In [ ]:
dbutils.widgets.text("p_fecha_proceso", "")
dbutils.widgets.text("p_catalogo", "")

var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")
var_catalogo = dbutils.widgets.get("p_catalogo")

logger = logging.getLogger("ETL_TRANSACCIONES")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_TRANSACCIONES")
logger.info("Parametros recibidos por widgets: fecha=%s catalogo=%s",
            var_fecha_proceso, var_catalogo)

## 4. Seccion constantes
> **Descripcion:** Valores que se mantienen constantes a lo largo del proceso.

In [ ]:
TBL_TRANSACCION_ORIGEN = f"{var_catalogo}.can.h_transaccion"
TBL_TRANSACCION_FINAL = f"{var_catalogo}.canales.fct_transaccion"

EST_APROBADA = "APROBADA"

## 5. Funciones de transformacion
> **Descripcion:** Funciones modularizadas de lectura, transformacion y escritura.

In [ ]:
def read_transaccion(tabla, fecha):
    """Lee las transacciones del periodo indicado."""
    return spark.sql(f"select * from {tabla} where fec_transaccion = '{fecha}'")


def add_canal_normalizado(df_origen):
    """Normaliza el codigo de canal a mayusculas."""
    return df_origen.withColumn("cod_canal", F.upper(F.col("cod_canal")))

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas previamente.

In [ ]:
ini_etapa = time.perf_counter()

df_transaccion = read_transaccion(TBL_TRANSACCION_ORIGEN, var_fecha_proceso)
df_transaccion_final = add_canal_normalizado(df_transaccion)

cant_registros = df_transaccion_final.count()
mto_maximo = df_transaccion_final.select(F.max("mto_transaccion")).collect()[0][0]

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Deduplicacion
> **Descripcion:** Logica de deduplicacion segun las llaves de la tabla.

In [ ]:
ventana_transaccion = Window.partitionBy("cod_transaccion").orderBy(
    F.col("fec_transaccion").desc()
)

df_transaccion_unica = (
    df_transaccion_final
    .withColumn("nro_orden", F.row_number().over(ventana_transaccion))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 8. Escritura en la tabla final
> **Descripcion:** Persistencia del resultado en formato Delta.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_transaccion_unica
        .write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(TBL_TRANSACCION_FINAL)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_TRANSACCION_FINAL, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 9. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con el registro de duracion y volumen.

In [ ]:
logger.info("Fin del proceso ETL_TRANSACCIONES. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)